In [ ]:
from astropy.io import ascii
from astropy.table import QTable,join,Column
from astropy.coordinates import SkyCoord, Galactocentric, Angle
from astropy.visualization import quantity_support
import hdbscan
import seaborn as sns
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from zero_point import zpt
import matplotlib.ticker as ticker
from astropy.coordinates import SkyCoord
from astropy.table import QTable
import os
from ipywidgets import interact, IntSlider
from IPython.display import display, clear_output
import matplotlib.ticker as ticker
import random
from astropy.stats import sigma_clip
zpt.load_tables()
quantity_support()
%matplotlib inline
%config InlineBackend.figure_format ='retina'

In [ ]:
from COSMIC_aux import *

In [ ]:
def aux_process(data):
    for column_name in data.colnames:
        if hasattr(data[column_name], 'mask'):  # Check if the column has a mask attribute
            # Convert column to float if it's an integer type
            if np.issubdtype(data[column_name].dtype, np.integer):
                data[column_name] = data[column_name].astype(float)
            # Now safe to fill masked values with np.nan
            data[column_name] = data[column_name].filled(np.nan)
    
    num_fidelity_numeric = np.sum(np.isfinite(data['fidelity_v2']))
    total_sources = len(data)
    print(f'The data contains {total_sources} sources')
    if total_sources == num_fidelity_numeric:
        print('All the sources have fidelity')
    else:
        print(f'Only {num_fidelity_numeric} sources have fidelity')
    print(f"There's {np.sum(~np.isnan(data['tmass_oid']))} of {total_sources} with 2MASS data")
    
    fidelity_selection = (0.5 < data['fidelity_v2'])
    data = data[fidelity_selection]
    print(f"{len(data)} remaining, {round(len(data)/total_sources*100)}% of good sources, with {np.sum(~np.isnan(data['tmass_oid']))} sources with 2MASS data")
    
    # Edit table columns
    data.rename_columns(['phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp'], ['Gmag', 'G_BPmag', 'G_RPmag', 'BP_RP'])
    columns_to_check = ['ra', 'dec', 'pmra', 'pmdec', 'parallax', 'Gmag', 'G_BPmag', 'G_RPmag']
    mask = np.zeros(len(data), dtype=bool)
    for column in columns_to_check:
        mask |= ~np.isfinite(data[column])
    # Apply the mask to the QTable
    data = data[~mask]
    print(len(data))
    np.sum(np.isnan(data['tmass_oid']))
    
    # Zero-Point Parallax
    data.rename_columns(['parallax'], ['parallax_observed'])
    data['zpvals'] = zpt.get_zpt(data['Gmag'], data['nu_eff_used_in_astrometry'], data['pseudocolour'], data['ecl_lat'], data['astrometric_params_solved']) * u.mas
    data['zpvals'] = np.ma.masked_invalid(data['zpvals']).filled(0)
    data['parallax'] = data['parallax_observed'] - data['zpvals']
    data['pmra_obs'], data['pmdec_obs'] = data['pmra'], data['pmdec']
    for i in data:
        i['pmra'], i['pmdec'] = edr3ToICRF(i['pmra_obs'].value, i['pmdec_obs'].value, i['ra'].value, i['dec'].value, i['Gmag'].value) * (u.mas / u.yr)
    
    # Apply the error functions to the respective G column to create new error columns with units
    data['e_Gmag'] = [g_mag_error(G) for G in data['Gmag']]
    data['e_G_BPmag'] = [g_bp_error(G) for G in data['G_BPmag']]
    data['e_G_RPmag'] = [g_rp_error(G) for G in data['G_RPmag']]
    
    # Calculate e_BP_RP as the square root of the sum of squares of e_G_BPmag and e_G_RPmag, correctly applying units
    data['e_BP_RP'] = np.sqrt(data['e_G_BPmag']**2 + data['e_G_RPmag']**2)
    data['e_J_H'] = np.sqrt(data['j_msigcom']**2 + data['h_msigcom']**2)
    data['e_RP_J'] = np.sqrt(data['e_G_RPmag']**2 + data['j_msigcom']**2)
    data['e_H_K'] = np.sqrt(data['h_msigcom']**2 + data['ks_msigcom']**2)
    data['e_BP_J'] = np.sqrt(data['e_G_BPmag']**2 + data['j_msigcom']**2)
    add_photometric_errors(data)
    data['RP_J'] = data['G_RPmag'] - data['j_m']
    data['H_K'] = data['h_m'] - data['ks_m']
    data['J_K'] = data['j_m'] - data['ks_m']
    data['J_H'] = data['j_m'] - data['h_m']
    data['BP_J'] = data['G_BPmag'] - data['j_m']
    return data

In [ ]:
def hdbscan_clustering(mcs, ms, data): 
    clusterer = hdbscan.HDBSCAN(
        algorithm='best',
        cluster_selection_method='eom',
        allow_single_cluster=False,
        min_cluster_size=mcs,
        min_samples=ms,
        core_dist_n_jobs=-1,
        gen_min_span_tree=True,
        metric='euclidean',
        match_reference_implementation=True).fit(data[['pmra', 'pmdec']].to_pandas())
    
    # Create a new array for modified cluster labels based on probabilities

    modified_labels = np.array(clusterer.labels_)
    modified_labels[clusterer.probabilities_ <= 0.5] = -1
    unique_labels, counts = np.unique(modified_labels, return_counts=True)
    n_clusters = len(unique_labels)
    max_label = max(unique_labels) if len(unique_labels) > 0 else 0
    color_palette = sns.color_palette('bright', max(max_label + 1, 10))  # Ensure a minimum palette size
    cluster_colors = [color_palette[x] if x >= 0 else (0.5, 0.5, 0.5) for x in modified_labels]
    cluster_member_colors = [sns.desaturate(x, p) for x, p in zip(cluster_colors, clusterer.probabilities_)]

    data['cluster'] = modified_labels
    data['probability'] = clusterer.probabilities_
    data['cluster_members_colors'] = cluster_member_colors
    fig, ax = plt.subplots(1, 4, layout='constrained', figsize=(20, 6))
    ax[0].scatter(data['pmra'][data['cluster'] != -1], data['pmdec'][data['cluster'] != -1], s=3, c=data['cluster_members_colors'][data['cluster'] != -1],alpha=0.95,zorder=10)
    ax[0].scatter(data['pmra'][data['cluster'] == -1], data['pmdec'][data['cluster'] == -1], s=3, c=data['cluster_members_colors'][data['cluster'] == -1],alpha=0.5,zorder=-1)
    ax[0].set_xlabel('PMRA',fontsize=16)
    ax[0].set_ylabel('PMDEC',fontsize=16)
    ax[0].set_aspect('equal')

    ax[1].scatter(data['BP_RP'][data['cluster'] != -1], data['Gmag'][data['cluster'] != -1], s=3, c=data['cluster_members_colors'][data['cluster'] != -1],alpha=0.95,zorder=10)
    ax[1].scatter(data['BP_RP'][data['cluster'] == -1], data['Gmag'][data['cluster'] == -1], s=3, c=data['cluster_members_colors'][data['cluster'] == -1],alpha=0.5,zorder=-1)
    ax[1].set_xlabel(r'$G_{BP} - G_{RP}$',fontsize=16)
    ax[1].set_ylabel(r'$G_{\text{mag}}$',fontsize=16)
    ax[1].invert_yaxis()

    ax[2].scatter(data['parallax'][data['cluster'] != -1], data['Gmag'][data['cluster'] != -1], s=3, c=data['cluster_members_colors'][data['cluster'] != -1],alpha=0.95,zorder=10)
    ax[2].scatter(data['parallax'][data['cluster'] == -1], data['Gmag'][data['cluster'] == -1], s=3, c=data['cluster_members_colors'][data['cluster'] == -1],alpha=0.5,zorder=-1)
    ax[2].set_xlabel(r'$\varpi$',fontsize=16)
    ax[2].set_ylabel(r'$G_{\text{mag}}$',fontsize=16)
    ax[2].invert_yaxis()

    clusterer.condensed_tree_.plot(select_clusters=True, selection_palette=color_palette, cmap=sns.color_palette("mako", as_cmap=True), axis=ax[3])
    ax[3].set_ylabel(r'$\lambda$ value', fontsize=16)
    
    # Subtitle text with mcs and ms included
    # Add mcs and ms to the subtitle text
    subtitle_text = f'Minimum cluster size : {mcs}, minimum sample: {ms}\n' + '\n'.join([
        f'Noise: {count} members, persistence {persistence:.2f}' if label == -1 
        else f'Cluster {label}: {count} members, persistence {persistence:.2f}'
        for label, count, persistence in zip(unique_labels, counts, np.insert(clusterer.cluster_persistence_, 0, 0))
    ])
    fig.suptitle(subtitle_text)

# Example usage:
# fig = plt.figure()
# fig.suptitle(subtitle_text)
# plt.show()

    for i in ax.flatten():
        i.xaxis.set_minor_locator(ticker.AutoMinorLocator())
        i.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        i.tick_params(axis='both', which='both', direction='in')
    
    plt.show()

In [ ]:
base_path = os.path.join(os.getcwd(), 'CLUSTER_DATA')
figures_path = os.path.join(os.getcwd(), 'Figures')

# Create the /Figures directory if it doesn't exist
if not os.path.exists(figures_path):
    os.makedirs(figures_path)

# List all .ecsv files in the CLUSTER_DATA directory excluding those with '_clustered'
file_list = []
for cluster_name in os.listdir(base_path):
    cluster_dir = os.path.join(base_path, cluster_name)
    if os.path.isdir(cluster_dir):
        for file_name in os.listdir(cluster_dir):
            if file_name.endswith('.ecsv') and '_clustered' not in file_name:
                file_path = os.path.join(cluster_dir, file_name)
                file_list.append(file_path)

In [ ]:
file_path = file_list[0]
file_name = os.path.basename(file_path)
try:
    data = QTable.read(file_path, guess=False, format='ascii.ecsv')
    print(f"Successfully read {file_name}")
except Exception as e:
    print(f"Failed to read {file_name}: {e}")
else:
    data = aux_process(data)  

In [ ]:
# Create an interactive plot with a slider for the number of points
interact(lambda mcs, ms: hdbscan_clustering(mcs, ms, data),  mcs=IntSlider(min=10, max=100, step=1, value=10, description='Min cluster size'), ms=IntSlider(min=10, max=100, step=1, value=10, description='Sample Size'));

In [ ]:
file_path = file_list[-1]
file_name = os.path.basename(file_path)
try:
    data_sure = QTable.read(file_path, guess=False, format='ascii.ecsv')
    print(f"Successfully read {file_name}")
except Exception as e:
    print(f"Failed to read {file_name}: {e}")
else:
    data_sure = aux_process(data_sure) 

In [ ]:
interact(lambda mcs, ms: hdbscan_clustering(mcs, ms, data_sure),  mcs=IntSlider(min=10, max=100, step=1, value=10, description='Number of Points'), ms=IntSlider(min=10, max=100, step=1, value=10, description='Sample Size'));

In [ ]:
file_path = file_list[-2]
file_name = os.path.basename(file_path)
try:
    data_sure2 = QTable.read(file_path, guess=False, format='ascii.ecsv')
    print(f"Successfully read {file_name}")
except Exception as e:
    print(f"Failed to read {file_name}: {e}")
else:
    data_sure2 = aux_process(data_sure2)

In [ ]:
interact(lambda mcs, ms: hdbscan_clustering(mcs, ms, data_sure2),  mcs=IntSlider(min=10, max=100, step=1, value=10, description='Number of Points'), ms=IntSlider(min=10, max=100, step=1, value=10, description='Sample Size'));

In [ ]:
    data['cluster'] = clusterer.labels_
    data['probability_hdbscan'] = clusterer.probabilities_
    data['probability_times'] = cluster_probabilities
    data['probability'] = data['probability_hdbscan'] * data['probability_times']
    color_palette = sns.color_palette('bright', len(np.unique(clusterer.labels_)))
    data['outlier_score'] = clusterer.outlier_scores_
    # Calculate the size of each cluster in data_qtable
    # QTable does not have the value_counts method, so we'll use a different approach
    clusters, counts = np.unique(data['cluster'], return_counts=True)
    # Find the cluster(s) with the same size as the desired cluster size
    matching_clusters = clusters[counts == desired_len]
    # For simplicity, assuming there's only one matching cluster and getting its name
    desired_cluster_name_qtable = matching_clusters[0] if len(matching_clusters) > 0 else None
    desired_cluster_name_qtable, len(matching_clusters)  # Returning the cluster name and the number of matching clusters
    data['cluster_hdbscan'] = data['cluster']
    data['cluster'] = np.where(data['cluster'] == desired_cluster_name_qtable, 
                               data['cluster'], 
                               -1)
    #ascii.write(data, f"{file_path}_clustered.ecsv", format='ecsv', overwrite=True)  # Create the clustered file

In [ ]:
# Define the base path
base_path = os.path.join(os.getcwd(), 'CLUSTER_DATA')
figures_path = os.path.join(os.getcwd(), 'Figures')

# Create the /Figures directory if it doesn't exist
if not os.path.exists(figures_path):
    os.makedirs(figures_path)

# List all .ecsv files in the CLUSTER_DATA directory excluding those with '_clustered'
file_list = []
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith('.ecsv') and '_clustered    n, effective_mcs, lambda_value, rv, cp = [], [], [], [], []
    labels_storage = {I: [] for I in range(len(data))}
    min_cluster_size_samples = np.arange(10, 300, 1)
    
    for i in min_cluster_size_samples:
        clusterer = hdbscan.HDBSCAN(
            algorithm='best',
            cluster_selection_method='eom',
            min_cluster_size=i,
            min_samples=30,
            allow_single_cluster=False,
            core_dist_n_jobs=-1,
            gen_min_span_tree=True,
            metric='euclidean',
            match_reference_implementation=True
        ).fit(data['pmra', 'pmdec'].to_pandas())

        for idx, label in enumerate(clusterer.labels_):
            labels_storage[idx].append(label)
        
        tree = clusterer.condensed_tree_.to_pandas()
        cluster_probabilities = np.array([np.count_nonzero(np.array(labels) != -1) / len(labels) for labels in labels_storage.values()])
        data['cluster'] = clusterer.labels_
        data['probability_hdbscan'] = clusterer.probabilities_
        data['probability_times'] = cluster_probabilities
        data['probability'] = data['probability_hdbscan'] * data['probability_times']

        if tree["lambda_val"].max() >= 1:
            max_lambda_val_row = tree["lambda_val"].idxmax()
            desired_parent = tree.at[max_lambda_val_row, "parent"]
            desired_len = len(tree[(tree["parent"] == desired_parent)])
            if len(np.unique(clusterer.labels_)) > 1 and len(data[data['probability'] >= 0.6]) >= 1:
                n.append(desired_len)
                effective_mcs.append(i)
                lambda_value.append(tree["lambda_val"].max())
                rv.append(clusterer.relative_validity_)
                cp.append(clusterer.cluster_persistence_)

    max_n_value = max(n)
    max_n_index = n.index(max_n_value)
    max_min_cluster_size = effective_mcs[max_n_index]
    
    max_lambda_value = max(lambda_value)
    max_lambda_index = lambda_value.index(max_lambda_value)
    max_lambda_min_cluster_size = effective_mcs[max_lambda_index]

    fig, ax = plt.subplots(1, 6, layout='constrained', figsize=(26, 6))
    
    ax[0].plot(effective_mcs, n)
    ax[0].axvline(max_min_cluster_size, color='b', linestyle='--', label=f'Optimal Min Cluster Size: {max_min_cluster_size}')
    ax[0].axhline(y=max_n_value, color='g', linestyle='--', label=f'Max Cluster Size: {max_n_value}')
    ax[0].set_xlabel('Min Cluster Size', fontsize=16)
    ax[0].set_ylabel('Cluster Size', fontsize=16)
    ax[0].legend()

    ax[1].plot(effective_mcs, rv)
    ax[1].axvline(max_min_cluster_size, color='b', linestyle='--', label=f'Optimal Min Cluster Size: {max_min_cluster_size}')
    ax[1].set_xlabel('Min cluster size', fontsize=16)
    ax[1].set_ylabel('Relative validity', fontsize=16)
    ax[1].legend()

    ax[2].axvline(max_min_cluster_size, color='b', linestyle='--', label=f'Optimal Min Cluster Size: {max_min_cluster_size}')
    ax[2].set_xlabel('Min cluster size', fontsize=16)
    ax[2].set_ylabel('Relative validity', fontsize=16)
    ax[2].legend()
    

    color_palette = sns.color_palette('bright', len(np.unique(clusterer.labels_)))
    cluster_colors = [color_palette[x] if x >= 0 else (0.5, 0.5, 0.5) for x in clusterer.labels_]
    cluster_member_colors = [sns.desaturate(x, p) for x, p in zip(cluster_colors, clusterer.probabilities_)]

    tree = clusterer.condensed_tree_.to_pandas()
    max_lambda_val_row = tree["lambda_val"].idxmax()
    desired_parent = tree.at[max_lambda_val_row, "parent"]
    desired_len = len(tree[tree["parent"] == desired_parent])


    plt.show()' not in file:
            file_list.append(os.path.join(root, file))

# Read all data files in the folder and store them in a list of QTables
data_tables = []